In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
!gcloud config set project YOUR_GCP_PROJECT_ID
%cd /content/drive/MyDrive/E231project
!ls benchmarks infra scripts config.yaml

In [ ]:
# Keep a local copy of the ETL scaling results (this run writes to the same results/ folder)
!mkdir -p results/etl_final && cp results/*etl* results/scaling_results.csv results/etl_final/ 2>/dev/null; ls results/etl_final

In [ ]:
# Converted raw Parquet must exist (made by EtlScalingRuns); canonical processed data may not yet
!gcloud storage ls gs://YOUR_BUCKET/raw_parquet/amazon_reviews_2018/
!gcloud storage ls gs://YOUR_BUCKET/full/processed/

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_launch.sh 16

In [ ]:
# ONE-TIME: build the canonical training data from all raw data (skip if full/processed/ already has train.parquet)
!zip -q -r /tmp/code.zip etl centralized evaluation federated benchmarks
!gcloud storage cp config.yaml gs://YOUR_BUCKET/full/config.yaml
P = "spark.dynamicAllocation.enabled=false,spark.executor.instances=16,spark.executor.cores=3,spark.executor.memory=9g"
!gcloud dataproc jobs submit pyspark scripts/run_etl.py --cluster=da231o --region=asia-south1 --py-files=/tmp/code.zip --properties={P} -- --config=gs://YOUR_BUCKET/full/config.yaml --step=build 2>&1 | grep -E "^\[etl|Traceback|Error|state:"

In [ ]:
!python scripts/run_benchmarks.py --config config.yaml --modes centralized federated

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_delete.sh